### Import libraries and Load Dataset

In [8]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from pathlib import Path
import json, joblib, numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

test_df = pd.read_csv(
    ROOT / "datasets/processed/paddy_test.csv",
    keep_default_na=False,
    encoding="utf-8-sig",
)

selection_path = ROOT / "models/model_evaluation_results/selected_model.json"
selection = json.loads(selection_path.read_text(encoding="utf-8"))
model_name = selection["selected_model"]

### Load Model

In [9]:
if model_name == "tfidf_linear_svc":
    model = joblib.load(
        ROOT / "models/problem_classifier/tfidf_linear_svc.joblib"
    )
    pred = model.predict(test_df["question"])
elif model_name == "xlm_roberta_base":
    model_dir = ROOT / "models/problem_classifier/xlm_roberta_base/best_model"

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
else:
    raise ValueError(f"Unsupported selected model: {model_name}")

### Prepare the test questions

In [10]:
if model_name == "xlm_roberta_base":
    infer_ds = Dataset.from_pandas(
        test_df[["question"]],
        preserve_index=False
    ).map(
        lambda x: tokenizer(
            x["question"],
            truncation=True,
            max_length=160
        ),
        batched=True
    )

### Generate predictions

In [11]:
if model_name == "xlm_roberta_base":
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=str(ROOT / "models/tmp_test_eval"),
            per_device_eval_batch_size=8,
            report_to="none"
        ),
        data_collator=DataCollatorWithPadding(tokenizer),
        processing_class=tokenizer
    )

    output = trainer.predict(infer_ds)
    id2label = {int(k): v for k, v in model.config.id2label.items()}
    pred = [
        id2label[int(i)]
        for i in np.argmax(output.predictions, axis=1)
    ]

### Calculate final performance metrics

In [12]:
p, r, f1, _ = precision_recall_fscore_support(
    test_df["label"],
    pred,
    average="macro",
    zero_division=0
)

metrics = {
    "selected_model": model_name,
    "accuracy": float(accuracy_score(test_df["label"], pred)),
    "precision_macro": float(p),
    "recall_macro": float(r),
    "f1_macro": float(f1)
}

print(json.dumps(metrics, indent=2))
print(classification_report(test_df["label"], pred, zero_division=0))

{
  "selected_model": "tfidf_linear_svc",
  "accuracy": 0.5760416666666667,
  "precision_macro": 0.5579906204906205,
  "recall_macro": 0.6430555555555556,
  "f1_macro": 0.5772942109780391
}
                         precision    recall  f1-score   support

     Disease management       0.25      0.25      0.25       480
Farm operations & costs       0.67      1.00      0.80       240
 Harvest & post-harvest       0.50      0.25      0.33       480
            Land & soil       0.67      1.00      0.80       240
     Nursery & planting       0.78      1.00      0.88       240
 Nutrients & fertilizer       0.50      0.50      0.50       240
        Pest management       0.50      0.25      0.33       480
   Safety & environment       0.67      1.00      0.80       240
         Seed & variety       1.00      0.72      0.83       240
       Water management       0.50      0.75      0.60       480
      Weather & climate       0.00      0.00      0.00       240
        Weed management      

### Save test predictions

In [13]:
predictions = test_df[
    ["qa_id", "pair_id", "question", "language", "label", "issue_id"]
].copy()

predictions["predicted_label"] = pred
predictions["correct"] = (
    predictions["label"] == predictions["predicted_label"]
)

In [14]:
result_dir = ROOT / "models/model_evaluation_results"
result_dir.mkdir(parents=True, exist_ok=True)

predictions.to_csv(
    result_dir / "final_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

(result_dir / "final_test_metrics.json").write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

189